# RQ1: matched input / decoder layer 16

Enable **Internet** and **T4 ×2**. Push the new files to `apet-analysis` first. Attach the **same private rq1-inputs-v1 dataset** used for the input/validity runs. No new input ZIP, audit upload or dependency versions are needed.

Start with **six frozen questions × 32 random positions × two stages = 384 patch/stage scores**. The panel is fixed before the running validity-test outcomes. Original ApET sources and its lockfile remain unchanged. The native decoder reconstruction routine runs on protected copies; its compressed output is discarded. Interventions observe the full uncompressed model after block 16, before block 17.

Import into an **editable draft**. Both flags start disabled to prevent accidental execution. Set `RUN_LAYER_CALIBRATION = True`, keep `RUN_FULL_LAYER_SWEEP = False`, and execute cells in order. Review the calibration and validity results before expanding. See `research/RQ1_LAYER_PILOT.md` for the exact protocol and existing-live-session instructions.

**Completed:** the six-case 576-position zeroing/IH comparison is archived. For the new 30-case input-stage IH extension, use `kaggle_rq1_input_pilot.ipynb` and `rq1-ih-inputs-v1.zip`; rerunning this layer panel is unnecessary. See `research/RQ1_INPUT_IH.md`.


In [ ]:
import json
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_LAYER_CALIBRATION = False  # Change to True for the first six-question x 32-position run.
RUN_IH_CALIBRATION = False     # New: True for zeroing + IH at layer 16; keep RUN_LAYER_CALIBRATION=False.
RUN_FULL_LAYER_SWEEP = False   # Later only: review calibration and validity checks first.
RUN_FULL_IH_SWEEP = False      # Later only: requires the new passed IH calibration.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None         # Optional exact ZIP path or extracted bundle directory.
CHECKPOINT_ARGS = []     # Optional --model / --vision-model local original-format snapshots.

if not (RUN_LAYER_CALIBRATION or RUN_IH_CALIBRATION):
    raise RuntimeError("Enable one calibration mode, Internet + T4 x2, then run cells manually.")
if RUN_LAYER_CALIBRATION and RUN_IH_CALIBRATION:
    raise RuntimeError("Choose one calibration mode: original zeroing or zeroing + IH.")
print("Preparing code; inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)
if not (repo / "scripts/run_rq1_layer.py").is_file():
    raise RuntimeError("Push the new layer-runner files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
inputs = Path("/kaggle/working/rq1-inputs-v1")
print("Repository:", repo, flush=True)


## Prepare inputs and the original runtime

The notebook kernel stays unchanged. Existing environments/checkpoint caches are reused. These prerequisite tests use tiny models and no downloaded weights. The explicit Torch import prevents silent test skips on Kaggle.


In [ ]:
if not (inputs / "protocol.json").exists():
    input_zip = Path(INPUT_ZIP) if INPUT_ZIP else None
    if input_zip is None:
        zip_candidates = list(Path("/kaggle/input").rglob("rq1-inputs-v1.zip"))
        zip_candidates += list(Path("/kaggle/working").glob("rq1-inputs-v1.zip"))
        extracted = []
        for path in Path("/kaggle/input").rglob("protocol.json"):
            try:
                if json.loads(path.read_text()).get("version") == "rq1-input-zero-v1":
                    if all((path.parent / name / "manifest.json").is_file() for name in ["gqa", "mmstar", "textvqa"]):
                        extracted.append(path.parent)
            except (ValueError, OSError):
                pass
        if len(zip_candidates) == 1:
            input_zip = zip_candidates[0]
        elif len(extracted) == 1:
            inputs = extracted[0]  # Kaggle may unpack a ZIP uploaded as a Dataset.
        else:
            raise RuntimeError(f"Attach rq1-inputs-v1.zip or its extracted files, and set INPUT_ZIP if needed. ZIPs: {zip_candidates}; folders: {extracted}")
    if input_zip is not None and input_zip.is_dir():
        inputs = input_zip
    elif input_zip is not None:
        inputs.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(input_zip) as z:
            for item in z.infolist():
                destination = (inputs / item.filename).resolve()
                if inputs.resolve() not in destination.parents:
                    raise ValueError("Unexpected ZIP path")
            z.extractall(inputs)
print("Accepted input directory:", inputs)
print(json.loads((inputs / "protocol.json").read_text()))
subprocess.run([
    "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
    "--output", "/kaggle/working/rq1-runtime.json",
], cwd=repo, check=True)
# Explicit import prevents Torch-dependent tests silently skipping on Kaggle.
subprocess.run([str(original_python), "-I", "-c", "import torch; print(torch.__version__)"], check=True)
tests = ["research/tests/test_rq1_layer.py", "research/tests/test_rq1_layer_native.py", "research/tests/test_rq1_native.py"]
if RUN_IH_CALIBRATION:
    tests += ["research/tests/test_rq1_layer_ih.py", "research/tests/test_rq1_layer_ih_native.py"]
for test in tests:
    subprocess.run([str(original_python), "-I", test], cwd=repo, check=True)


## Streaming helper

Every run uses a new timestamped directory, loads the model once for all six questions, and preserves a separate console log plus a ZIP even on a normally handled failure. Finish the currently running process before updating the checkout or starting another model.


In [ ]:
def run_layer(tokens=32, calibration=None, include_ih=False):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    name = "calibration-v1" if tokens == 32 else "sixcases-v1"
    method = "-ih" if include_ih else ""
    folder = Path(f"/kaggle/working/rq1-layer16{method}-{name}-{stamp}")
    log = folder.with_suffix(".log")
    script = "scripts/run_rq1_layer_ih.py" if include_ih else "scripts/run_rq1_layer.py"
    command = [str(original_python), "-X", "utf8", "-I", script,
               "--inputs", str(inputs), "--tokens", str(tokens), "--output", str(folder),
               "--download-budget-gib", "20", *CHECKPOINT_ARGS]
    if calibration is not None:
        command += ["--calibration-report", str(Path(calibration) / "report.json")]
    print("Run directory:", folder, flush=True)
    with log.open("x", encoding="utf-8") as stream:
        process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print("Interrupted. Preserve the log and any report/ZIP in", folder, flush=True)
            raise
        finally:
            process.stdout.close()
    if (folder / "report.json").exists():
        report = json.loads((folder / "report.json").read_text(encoding="utf-8"))
        print("Status:", report["status"], "wall seconds:", round(report["wall_seconds"], 1))
        print("Progress:", report.get("progress"))
        for question in report.get("questions", []):
            print(question)
        for question in report.get("ih_questions", []):
            print("IH comparison:", question)
        if report.get("error"):
            print("Failure:", report["error"])
    print("Download:", folder.with_suffix(".zip"), "and", log)
    if status:
        raise RuntimeError("Layer run failed; download its ZIP and log before changing anything.")
    return folder


## First: six-question, 32-position calibration

Each question has a fresh correct baseline, fixed target-ID check, input repeats/no-op/native parity, layer observation-only/no-op/repeated-state controls and final reference checks. Failures stop without retargeting. Positions are randomly chosen independently of E/I and match across stages.


In [ ]:
layer_calibration_dir = run_layer(include_ih=RUN_IH_CALIBRATION)


## Later only: full 576-position sweep

Leave both full flags disabled for the first run. Review the calibration ZIP/log first. The original zeroing sweep has 6,912 patch/stage scores. The IH extension adds 3,456 retain-one scores at layer 16 (10,368 total). `RUN_FULL_IH_SWEEP` requires the new passed IH calibration; the older zero-only calibration cannot unlock it. See `research/RQ1_LAYER_IH_PILOT.md` for the physical-deletion operator, 1x1 unit and first-token probability metrics.


In [ ]:
if RUN_FULL_IH_SWEEP:
    layer_full_dir = run_layer(tokens=576, calibration=layer_calibration_dir, include_ih=True)
elif RUN_FULL_LAYER_SWEEP:
    layer_full_dir = run_layer(tokens=576, calibration=layer_calibration_dir)
else:
    print("Full sweep disabled. Download and review calibration + validity evidence first.")


## Download before stopping the session

Download **both ZIP and log**. The HTML/CSV summary reports per-question and basis-aware rank comparisons; raw records include controls, native E/selection/merge diagnostics, target/position mappings, images, small feature snapshots and source/lock hashes. These curated questions are diagnostic, not population accuracy evidence.

Input and layer reconstruction errors inhabit different feature spaces; compare ranks rather than raw values. At layer 16 other states may already contain the patch information. Local layer zeroing is therefore not equivalent to removing that original input patch, and deeper visual E is not automatically conditioned on the later question.


In [ ]:
from IPython.display import FileLink, display
folders = [layer_calibration_dir]
if (RUN_FULL_LAYER_SWEEP or RUN_FULL_IH_SWEEP) and "layer_full_dir" in globals():
    folders.append(layer_full_dir)
for folder in folders:
    for path in [folder.with_suffix(".zip"), folder.with_suffix(".log")]:
        print(path)
        display(FileLink(str(path)))
